# 07 — Experiments (runnable)
**Author:** Elif Yegenoglu

Runnable versions of the key experiments behind the final model, rebuilt on the v5 pipeline so
anyone can reproduce the **decisions** with Run All. The original runs are preserved with their
outputs in `experiments.ipynb`; numbers here can differ slightly from those receipts (the v1
sample and feature set differed) — the *conclusions* should not.

E1 trivial baselines · E2 beliefs in/out · E3 individual items vs composites ·
E4 alternative Part 2 outcomes · E5 weighted refit · E6 tuning (slow — run last or skip)

> **Shared setup** — copied verbatim from `02_model.ipynb` (same rule as 03/04: if the pipeline changes there, re-copy here).

In [ ]:
# ============================================================
# CELL 0 — paths (portable: finds the repo by walking up from cwd)
# No editing needed on any machine. If it errors, open the repo
# folder itself in VS Code / Jupyter and restart the kernel.
# ============================================================
from pathlib import Path

def find_root(start=None, depth=6):
    p = start or Path.cwd()
    for _ in range(depth):
        if (p / "Data").exists() and (p / "Model").exists():
            return p
        p = p.parent
    raise FileNotFoundError(
        f"repo root not found walking up from {Path.cwd()} — "
        "in VS Code use File > Open Folder on summer26-teacher-ai-readiness, "
        "reopen this notebook, restart the kernel")

ROOT = find_root()
DATA_DIR = ROOT / "Data"                 # codebook + small CSVs
SPSS_DIR = DATA_DIR / "SPSS"             # raw TALIS .sav files (gitignored)
OUT_DIR  = DATA_DIR / "output"           # everything the notebooks produce (gitignored)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", ROOT)

In [ ]:
import matplotlib.pyplot as plt   # figures below need it (in 02 it arrives with the bake-off cell)

In [ ]:
# ============================================================
# CELL 1 — load merged file, build ai_sample
# ============================================================
import re
import numpy as np
import pandas as pd

merged = pd.read_csv(DATA_DIR / "output" / "teacher_principal_named_columns.csv",
                     encoding="utf-8-sig", low_memory=False)
assert any(c.startswith("P_TC") for c in merged.columns), "no principal columns - wrong file?"

q36_col = next(c for c in merged.columns if c.startswith("TT4G36"))
q36_num = pd.to_numeric(merged[q36_col], errors="coerce")
ai_sample = merged[q36_num != 8].copy()
ai_sample[q36_col] = pd.to_numeric(ai_sample[q36_col], errors="coerce").replace(9, np.nan)
print("administered the AI module:", len(ai_sample))

In [ ]:
# ============================================================
# CELL 2 — teacher features (RQ1) + prep_general + belief composites
# Definitions = official TALIS 2024 codebook labels (abbreviated)
# ============================================================
cb = pd.read_csv(DATA_DIR / "talis2024_teacher_codebook.csv")

def tcol(v):
    """short code -> actual teacher column name in merged/ai_sample"""
    return next(c for c in ai_sample.columns if c.startswith(v + " ") or c == v)

# --- AI-belief COMPOSITES (mean of items; code 5 "I don't know" -> missing) ---
# Q35 scale: 1=Strongly disagree ... 4=Strongly agree, 5=I don't know (NOT on scale)
def belief_mean(items):
    B = ai_sample[[tcol(v) for v in items]].apply(pd.to_numeric, errors='coerce')
    B = B.replace(5, 2.5)                       # ← THIS LINE: "don't know" (5) -> neutral 2.5
    B = B.where(B.isin([1, 2, 2.5, 3, 4]))      # ← note 2.5 added to valid values
    return B.mean(axis=1).where(B.notna().sum(axis=1) >= 1)
ai_sample['ai_benefit_mean'] = belief_mean([f'TT4G35{c}' for c in 'ABCDE'])  # benefits
ai_sample['ai_risk_mean']    = belief_mean([f'TT4G35{c}' for c in 'FGHIJ'])  # risks/concerns

# --- general pedagogical preparation composite ---
# Q7a-e,g: prepared for content / subject pedagogy / general pedagogy /
# classroom practice / multicultural settings / student development
prep_codes = ['TT4G07A', 'TT4G07B', 'TT4G07C', 'TT4G07D', 'TT4G07E', 'TT4G07G']
P = ai_sample[[tcol(v) for v in prep_codes]].apply(pd.to_numeric, errors='coerce')
P = P.where(~P.isin([6, 8, 9]))
ai_sample['prep_general'] = P.mean(axis=1).where(P.notna().sum(axis=1) >= 4)

feature_cols = (
    # --- AI / digital core ---
    ['TT4G21G',   # Q21g: professional learning included "using AI for teaching and learning" (yes/no)
     'TT4G07F',   # Q7f: felt prepared for "use of digital resources and tools" (initial education)
     'TT4G27M']   # Q27m: self-efficacy — support learning through digital resources and tools

    # --- AI beliefs: two COMPOSITE scores (replaces the 10 Q35 items) ---
    + ['ai_benefit_mean',  # mean of Q35A-E; higher = agrees AI is beneficial
       'ai_risk_mean']     # mean of Q35F-J; higher = agrees AI is risky/concerning

    # --- derived: general pedagogical preparation ---
    + ['prep_general']

    # --- professional environment (TALIS derived scales) ---
    + ['T4COLES',   # Professional collaboration in lessons among teachers
       'T4TLEAD',   # Teacher leadership
       'T4VALP']    # Perceptions of value and policy influence
                    # NOTE: T4SELF (overall self-efficacy) dropped — overlaps with
                    # TT4G27M (digital self-efficacy), the AI-relevant facet we keep

    # --- wellbeing / stress (derived scales) ---
    + ['T4JOBSAT',  # Job satisfaction, overall
       'T4WLOADT',  # Workload stress
       'T4STBEH',   # Student behaviour stress
       'T4CHFAT']   # Change fatigue

    # --- workload ---
    + ['TT4G15']    # Q15: hours teaching at this school, most recent full week

    # --- demographics / employment (grouped) ---
    + ['T4TAGEGR',  # Teacher age (grouped)
       'T4TEMPWH',  # Employment status by working hours (grouped; CATEGORICAL)
       'T4TNSCH']   # Number of schools the teacher works at

    # --- structure ---
    + ['CNTRY']     # Country alpha code (CATEGORICAL, fixed effect)
)

withlabels = {
    'TT4G21G':         'Received AI Training',
    'ai_benefit_mean': 'AI-benefit beliefs',
    'ai_risk_mean':    'AI-risk beliefs',
    'CNTRY':           'Country',
    'TT4G27M':         'Digital self-efficacy',
    'TT4G07F':         'Digital preparedness',
    'prep_general':    'General preparation',
    'T4TAGEGR':        'Age group',
    'T4COLES':         'Professional collaboration',
    'T4TLEAD':         'Teacher leadership',
    'T4VALP':          'Perceived value & influence',
    'T4JOBSAT':        'Job satisfaction',
    'T4WLOADT':        'Workload stress',
    'T4STBEH':         'Student-behaviour stress',
    'T4CHFAT':         'Change fatigue',
    'TT4G15':          'Teaching hours',
    'T4TEMPWH':        'Employment status',
    'T4TNSCH':         'Number of schools',
}

assert len(feature_cols) == len(set(feature_cols))
print("teacher features:", len(feature_cols))   # 18

In [ ]:
# ============================================================
# CELL 4 — assembly: ONE fixed complete-case sample + accounting
# ============================================================
def col_for(v):
    if v in ai_sample.columns:          # prep_general, composites
        return v
    return tcol(v)

all_short = feature_cols #+ school_block           # teacher features only (school block is for subset analysis)
D = ai_sample[[col_for(v) for v in all_short]].copy()
D.columns = all_short

# per-variable missing codes from the codebook (raw teacher vars only)
cb_codes = {}
for _, row in cb[cb.variable_name.isin(feature_cols)].iterrows():
    cb_codes[row.variable_name] = [float(x) for x in
        re.findall(r"(\d+)\s*=", str(row.special_missing_or_skip_codes))]

categoricals = ['CNTRY', 'T4SCHLOC', 'T4TEMPWH']
DERIVED = ['prep_general', 'ai_benefit_mean', 'ai_risk_mean']   # ← already clean, skip
for v in feature_cols:
    if v in categoricals or v in DERIVED:                        # ← was: == 'prep_general'
        continue
    D[v] = pd.to_numeric(D[v], errors='coerce')
    if cb_codes.get(v):
        D[v] = D[v].where(~D[v].isin(cb_codes[v]))

D['TT4G21G'] = D['TT4G21G'].map({1: 1, 2: 0})     # yes/no -> 1/0

D['y'] = (pd.to_numeric(ai_sample[q36_col], errors='coerce') == 1).astype(float)
D.loc[ai_sample[q36_col].isna(), 'y'] = np.nan
D['IDSCHOOL'] = ai_sample[next(c for c in ai_sample.columns if c.startswith('IDSCHOOL'))]
D['CNTRY'] = D['CNTRY'].astype(str).str.strip()
D['T4TEMPWH'] = pd.to_numeric(D['T4TEMPWH'], errors='coerce').where(
    lambda s: ~s.isin([8, 9]))

# ---- missingness accounting ----
n0 = len(D)
print("worst 10 columns by missingness (%):")
print(D[all_short].isna().mean().mul(100).round(1).sort_values(ascending=False).head(10).to_string())

data = D.dropna()
print(f"\ncomplete cases: {len(data):,} of {n0:,}  ({len(data)/n0*100:.1f}%)")
print(f"AI-use rate: full {D['y'].mean():.3f}  |  complete-case {data['y'].mean():.3f}")

kept = pd.Series(D.index.isin(data.index), index=D.index)
drop_by_cntry = (1 - kept.groupby(D['CNTRY']).mean()).mul(100).round(1)
print("\ncountries losing the most rows (%):")
print(drop_by_cntry.sort_values(ascending=False).head(8).to_string())

for v in ['T4TAGEGR', 'TT4G21G']:
    print(f"{v}: kept mean {D.loc[kept, v].mean():.3f} | dropped mean {D.loc[~kept, v].mean():.3f}")

cc = data['CNTRY'].value_counts()
print("\nsmallest country cells:", cc.tail(3).to_dict())
print("countries with <200 rows:", (cc < 200).sum())

In [ ]:
# ============================================================
# CELL 6 — school-grouped split + helpers (used by all comparisons)
# ============================================================
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score
from scipy.stats import norm

RANDOM_STATE = 42
CATEGORICALS = ['CNTRY', 'T4TEMPWH']          # T4SCHLOC gone (school block out)

gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
tr_idx, te_idx = next(gss.split(data, data['y'], groups=data['IDSCHOOL']))
train, test = data.iloc[tr_idx], data.iloc[te_idx]
assert not (set(train['IDSCHOOL']) & set(test['IDSCHOOL'])), "schools leak!"
print(f"train {len(train):,} | test {len(test):,}")
print(f"AI share — train {train['y'].mean():.3f} | test {test['y'].mean():.3f}")

def make_pipe(feats, clf=None):
    cat = [c for c in feats if c in CATEGORICALS]
    num = [c for c in feats if c not in CATEGORICALS]
    steps = [('num', StandardScaler(), num)]
    if cat:
        steps.append(('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat))
    if clf is None:
        clf = GradientBoostingClassifier(random_state=RANDOM_STATE)
    return Pipeline([('prep', ColumnTransformer(steps)), ('clf', clf)])


HEADLINE_CLF = lambda: GradientBoostingClassifier(random_state=RANDOM_STATE)

def delong(y, p1, p2):
    y = np.asarray(y); pos, neg = y == 1, y == 0
    m, n = pos.sum(), neg.sum()
    def struct(p):
        x, yv = p[pos], p[neg]
        v10 = np.array([(np.sum(xi > yv) + .5*np.sum(xi == yv))/n for xi in x])
        v01 = np.array([(np.sum(x > yi) + .5*np.sum(x == yi))/m for yi in yv])
        return v10, v01, v10.mean()
    a1v,a1w,a1 = struct(np.asarray(p1)); a2v,a2w,a2 = struct(np.asarray(p2))
    s10 = np.cov(np.vstack([a1v,a2v])); s01 = np.cov(np.vstack([a1w,a2w]))
    var = (s10[0,0]+s10[1,1]-2*s10[0,1])/m + (s01[0,0]+s01[1,1]-2*s01[0,1])/n
    z = (a1-a2)/np.sqrt(var) if var > 0 else 0.0
    return a1, a2, 2*norm.sf(abs(z))

## E1 · Trivial baselines

In [ ]:
# ============================================================
# E1 — trivial baselines (Checkpoint 1 learnability test)
# v1 receipt: dummy 0.500, logistic 0.797 (on the v1 sample)
# ============================================================
from sklearn.linear_model import LogisticRegression

for nm, clf in [('Dummy (stratified)', DummyClassifier(strategy='stratified', random_state=RANDOM_STATE)),
                ('Logistic',           LogisticRegression(max_iter=2000)),
                ('GradientBoosting',   HEADLINE_CLF())]:
    p = make_pipe(feature_cols, clf).fit(train[feature_cols], train['y']).predict_proba(test[feature_cols])[:, 1]
    print(f"{nm:20s} AUC {roc_auc_score(test['y'], p):.3f}")

## E2 · Do beliefs belong in the model?

In [ ]:
# ============================================================
# E2 — do beliefs belong? model WITH vs WITHOUT the composites
# v1 receipt: 0.846 vs 0.768 (+0.078); without beliefs, training soaks up everything
# ============================================================
from sklearn.inspection import permutation_importance

no_beliefs = [f for f in feature_cols if f not in ('ai_benefit_mean', 'ai_risk_mean')]

pipe_wb = make_pipe(feature_cols, HEADLINE_CLF()).fit(train[feature_cols], train['y'])
pipe_nb = make_pipe(no_beliefs,   HEADLINE_CLF()).fit(train[no_beliefs],   train['y'])
auc_wb = roc_auc_score(test['y'], pipe_wb.predict_proba(test[feature_cols])[:, 1])
auc_nb = roc_auc_score(test['y'], pipe_nb.predict_proba(test[no_beliefs])[:, 1])
print(f"WITH beliefs    AUC {auc_wb:.3f}  ({len(feature_cols)} features)")
print(f"WITHOUT beliefs AUC {auc_nb:.3f}  ({len(no_beliefs)} features)")
print(f"difference      {auc_wb - auc_nb:+.3f}\n")

r = permutation_importance(pipe_nb, test[no_beliefs], test['y'],
                           scoring='roc_auc', n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
print("top predictors WITHOUT beliefs (training dominates):")
print(pd.DataFrame({'feature': no_beliefs, 'auc_drop': r.importances_mean})
        .sort_values('auc_drop', ascending=False).head(8).to_string(index=False))

## E3 · Individual belief items vs the two composites

In [ ]:
# ============================================================
# E3 — 10 individual TT4G35 items instead of the 2 composites
# v3 receipt: belief signal scatters across collinear items
# Same recode as the composites: "don't know" (5) -> 2.5 midpoint
# ============================================================
from sklearn.inspection import permutation_importance

ITEMS = [f'TT4G35{c}' for c in 'ABCDEFGHIJ']
IT = pd.DataFrame({v: pd.to_numeric(ai_sample[tcol(v)], errors='coerce') for v in ITEMS})
IT = IT.replace(5, 2.5)
IT = IT.where(IT.isin([1, 2, 2.5, 3, 4]))

base = [f for f in feature_cols if f not in ('ai_benefit_mean', 'ai_risk_mean')]
di = data[base + ['y', 'IDSCHOOL']].join(IT).dropna()
print(f"complete cases with all 10 items: {len(di):,} (composites sample: {len(data):,})")

tri, tei = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
                .split(di, di['y'], groups=di['IDSCHOOL']))
tr_i, te_i = di.iloc[tri], di.iloc[tei]

feats_items = base + ITEMS
pipe_i = make_pipe(feats_items, HEADLINE_CLF()).fit(tr_i[feats_items], tr_i['y'])
auc_i = roc_auc_score(te_i['y'], pipe_i.predict_proba(te_i[feats_items])[:, 1])

# composites model on the SAME rows, for a fair comparison
pipe_c = make_pipe(feature_cols, HEADLINE_CLF()).fit(
    data.loc[tr_i.index, feature_cols], data.loc[tr_i.index, 'y'])
auc_c = roc_auc_score(data.loc[te_i.index, 'y'],
                      pipe_c.predict_proba(data.loc[te_i.index, feature_cols])[:, 1])
print(f"10 individual items AUC {auc_i:.3f} ({len(feats_items)} features)")
print(f"2 composites        AUC {auc_c:.3f} ({len(feature_cols)} features)\n")

r = permutation_importance(pipe_i, te_i[feats_items], te_i['y'],
                           scoring='roc_auc', n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
print("item-level importance (signal scatters across siblings):")
print(pd.DataFrame({'feature': feats_items, 'auc_drop': r.importances_mean})
        .sort_values('auc_drop', ascending=False).head(12).to_string(index=False))

> *Setup for Part 2 (target + sample + split, copied from the Part 2 cell in `02_model.ipynb`).*

In [ ]:
# ============================================================
# CELL 9 — Part 2 target: student-facing AI use among users
# ============================================================
from sklearn.inspection import permutation_importance

STUDENT_FACING = ['TT4G37A', 'TT4G37G', 'TT4G37H']   # assess/mark, student data, practice
TEACHER_FACING = ['TT4G37B', 'TT4G37C', 'TT4G37F']   # summarise, lesson plans, comms
AMBIGUOUS      = ['TT4G37D', 'TT4G37E']              # SEN, auto-adjust -> sensitivity only
ALL37 = STUDENT_FACING + TEACHER_FACING + AMBIGUOUS

p2 = ai_sample[pd.to_numeric(ai_sample[q36_col], errors='coerce') == 1].copy()
for c in ALL37:
    p2[c] = pd.to_numeric(p2[tcol(c)], errors='coerce').map({1: 1, 2: 0})
p2 = p2.dropna(subset=STUDENT_FACING + TEACHER_FACING)   # must answer all 6 core items
print(f"AI users with complete purpose data: {len(p2):,}")

p2['y_core']  = p2[STUDENT_FACING].max(axis=1).astype(int)
p2['y_broad'] = p2[STUDENT_FACING + AMBIGUOUS].max(axis=1).fillna(p2['y_core']).astype(int)
print(f"base rate y_core:  {p2['y_core'].mean():.3f}")
print(f"base rate y_broad: {p2['y_broad'].mean():.3f}")
flipped = (p2['y_core'] != p2['y_broad']).mean()
print(f"y_broad flips {flipped*100:.1f}% of labels (sensitivity check)")

# teacher-facing-only (the holdout group): student-facing = 0
teacher_only = ((p2['y_core'] == 0) & (p2[TEACHER_FACING].max(axis=1) == 1)).mean()
print(f"teacher-facing only (no student-facing use): {teacher_only*100:.1f}%")
# ============================================================
# CELL 10 — Part 2 sample + grouped split (teacher features only)
# ============================================================
all_short = feature_cols                               # teacher-only, matches Part 1
D2 = p2[[col_for(v) for v in all_short]].copy()
D2.columns = all_short
for v in feature_cols:
    if v in CATEGORICALS or v in ('prep_general', 'ai_benefit_mean', 'ai_risk_mean'):
        continue
    D2[v] = pd.to_numeric(D2[v], errors='coerce')
    if cb_codes.get(v):
        D2[v] = D2[v].where(~D2[v].isin(cb_codes[v]))
D2['TT4G21G'] = D2['TT4G21G'].map({1: 1, 2: 0})
D2['y'] = p2['y_core'].values
D2['IDSCHOOL'] = p2[next(c for c in p2.columns if c.startswith('IDSCHOOL'))].values
D2['CNTRY'] = D2['CNTRY'].astype(str).str.strip()
D2['T4TEMPWH'] = pd.to_numeric(D2['T4TEMPWH'], errors='coerce')
D2.loc[D2['T4TEMPWH'].isin([8, 9]), 'T4TEMPWH'] = np.nan

data2 = D2.dropna()
print(f"Part 2 complete cases: {len(data2):,} ({len(data2)/len(D2)*100:.1f}%)")
print(f"positive rate: {data2['y'].mean():.3f} | trained share: {data2['TT4G21G'].mean():.3f}")

tri2, tei2 = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
                  .split(data2, data2['y'], groups=data2['IDSCHOOL']))
train2, test2 = data2.iloc[tri2], data2.iloc[tei2]
assert not (set(train2['IDSCHOOL']) & set(test2['IDSCHOOL']))
print(f"train {len(train2):,} | test {len(test2):,}")

## E4 · Alternative Part 2 outcome definitions

In [ ]:
# ============================================================
# E4 — alternative Part 2 outcome definitions
# v1 receipts: 'leans pedagogical' 0.623, 'heavy pedagogical' 0.781
# Rebuilt with the final taxonomy (student-facing = A/G/H) on the v5 sample.
# Requires: p2, data2, train2, test2 (Part 2 setup cell above)
# ============================================================
core_items = p2.loc[data2.index, STUDENT_FACING]
tf_items   = p2.loc[data2.index, TEACHER_FACING]

alts = {
    'y_core (final): any student-facing':   data2['y'],
    'leans student-facing (SF > TF count)': (core_items.sum(axis=1) > tf_items.sum(axis=1)).astype(int),
    '2+ student-facing items':              (core_items.sum(axis=1) >= 2).astype(int),
}
for nm, yv in alts.items():
    ytr, yte = yv.loc[train2.index], yv.loc[test2.index]
    if ytr.nunique() < 2:
        print(f"{nm:42s} degenerate target, skipped"); continue
    p = (make_pipe(feature_cols, HEADLINE_CLF()).fit(train2[feature_cols], ytr)
         .predict_proba(test2[feature_cols])[:, 1])
    print(f"{nm:42s} base rate {yv.mean():.3f} | AUC {roc_auc_score(yte, p):.3f}")

## E5 · Weighted refit

In [ ]:
# ============================================================
# E5 — weighted refit: does TCHWGT change the model?
# v1 receipt: weighted 0.842 vs unweighted 0.846
# ============================================================
wcol = next(c for c in ai_sample.columns if c.split(' ')[0] == 'TCHWGT')
w = pd.to_numeric(ai_sample[wcol], errors='coerce').reindex(data.index)

pipe_u = make_pipe(feature_cols, HEADLINE_CLF()).fit(train[feature_cols], train['y'])
pipe_w = make_pipe(feature_cols, HEADLINE_CLF()).fit(
    train[feature_cols], train['y'],
    clf__sample_weight=w.loc[train.index].fillna(w.median()).values)

print(f"unweighted AUC {roc_auc_score(test['y'], pipe_u.predict_proba(test[feature_cols])[:, 1]):.3f}")
print(f"weighted   AUC {roc_auc_score(test['y'], pipe_w.predict_proba(test[feature_cols])[:, 1]):.3f}")

## E6 · Hyperparameter tuning — SLOW, run last or skip

In [ ]:
# ============================================================
# E6 — hyperparameter tuning (SLOW: 12 candidates x 3 folds — run last or skip)
# v1 receipt: tuned 0.849 vs untuned 0.846 -> kept defaults
# ============================================================
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold

param_dist = {
    'clf__n_estimators':  [100, 200, 300],
    'clf__learning_rate': [0.03, 0.05, 0.1],
    'clf__max_depth':     [2, 3, 4],
    'clf__subsample':     [0.8, 1.0],
}
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
search = RandomizedSearchCV(make_pipe(feature_cols, HEADLINE_CLF()), param_dist, n_iter=12,
                            cv=cv, scoring='roc_auc', random_state=RANDOM_STATE, n_jobs=-1)
search.fit(train[feature_cols], train['y'], groups=train['IDSCHOOL'])
print("best params:", search.best_params_)
print(f"best CV AUC: {search.best_score_:.3f}")
auc_t = roc_auc_score(test['y'], search.best_estimator_.predict_proba(test[feature_cols])[:, 1])
auc_d = roc_auc_score(test['y'],
    make_pipe(feature_cols, HEADLINE_CLF()).fit(train[feature_cols], train['y'])
        .predict_proba(test[feature_cols])[:, 1])
print(f"tuned test AUC {auc_t:.3f} | default test AUC {auc_d:.3f} | gain {auc_t-auc_d:+.3f}")